# 🎤 XTTS v2 Fine-Tuning — Pacaveli Voice Clone

Fine-tunes Coqui XTTS v2 on the Pacaveli voice dataset to produce **authentic timbre + delivery**.

| Step | What it does |
|------|-------------|
| 1 | Verify GPU (T4 required) |
| 2 | Install TTS + Whisper |
| 3 | Mount Drive, extract dataset |
| 4 | Convert audio → 22kHz WAV, transcribe with Whisper |
| 5 | Fine-tune XTTS v2 |
| 6 | Package & download `xtts_finetuned.zip` |
| 7 | Install instructions |

**Time**: ~30–90 min on Colab T4  
**After download**: click 📥 INSTALL MODEL in the app → select the `.zip`

In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

try:
    import torch
    print(f'PyTorch: {torch.__version__}')
    print(f'CUDA available: {torch.cuda.is_available()}')
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    else:
        print('⚠️  No GPU detected — switch Runtime → T4 GPU before continuing.')
        sys.exit(1)
except ImportError:
    print('PyTorch not installed yet — that is fine, Cell 2 will install it.')

In [ ]:
# ── Cell 2: Install TTS and Whisper ─────────────────────────────
# TTS installs Coqui TTS with XTTS v2 support
# openai-whisper is used to auto-transcribe the voice clips
!pip install TTS>=0.22.0 --no-cache-dir -q
!pip install openai-whisper --no-cache-dir -q
!pip install ffmpeg-python --no-cache-dir -q

# Verify
from TTS.api import TTS  # type: ignore
import whisper            # type: ignore
print('✅ TTS and Whisper installed')

In [ ]:
# ── Cell 3: Mount Google Drive and extract dataset ───────────────
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil
from pathlib import Path

# ── Locate the training zip ──────────────────────────────────────
# Try common names — change ZIP_PATH if yours is named differently
CANDIDATE_ZIPS = [
    '/content/drive/MyDrive/Pacaveli_vocals_training.zip',
    '/content/drive/MyDrive/Pacaveli_colab_training.zip',
    '/content/drive/MyDrive/pacaveli_training.zip',
]
ZIP_PATH = None
for z in CANDIDATE_ZIPS:
    if Path(z).exists():
        ZIP_PATH = z
        print(f'Found: {z}')
        break

if not ZIP_PATH:
    raise FileNotFoundError(
        'Training zip not found in Google Drive root.\n'
        'Upload one of: ' + ', '.join(CANDIDATE_ZIPS)
    )

EXTRACT_DIR = Path('/content/dataset_raw')
EXTRACT_DIR.mkdir(exist_ok=True)

print(f'Extracting {ZIP_PATH} …')
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(str(EXTRACT_DIR))

# Find the Pacaveli audio dir (handle different zip structures)
audio_dirs = [
    EXTRACT_DIR / 'dataset' / 'Pacaveli',
    EXTRACT_DIR / 'Pacaveli',
    EXTRACT_DIR,
]
AUDIO_DIR = None
for d in audio_dirs:
    files = list(d.glob('*.wav')) + list(d.glob('*.mp3')) if d.is_dir() else []
    if files:
        AUDIO_DIR = d
        print(f'Audio dir: {d}  ({len(files)} files)')
        break

if not AUDIO_DIR:
    raise FileNotFoundError('Could not find audio files inside the zip.')

In [ ]:
# ── Cell 4: Convert audio to 22kHz WAV + Whisper transcription ──
import subprocess, csv, random
from pathlib import Path
import whisper  # type: ignore

WAV_DIR = Path('/content/wavs')
WAV_DIR.mkdir(exist_ok=True)

# Convert all audio to 22050 Hz mono WAV (required by XTTS)
src_files = list(AUDIO_DIR.glob('*.wav')) + list(AUDIO_DIR.glob('*.mp3')) + \
            list(AUDIO_DIR.glob('*.flac')) + list(AUDIO_DIR.glob('*.m4a'))

print(f'Converting {len(src_files)} clips to 22kHz WAV…')
converted = []
for src in src_files:
    dst = WAV_DIR / (src.stem + '.wav')
    r = subprocess.run(
        ['ffmpeg', '-i', str(src), '-ar', '22050', '-ac', '1', '-y', str(dst)],
        capture_output=True
    )
    if dst.exists() and dst.stat().st_size > 1000:
        converted.append(dst)
    else:
        print(f'  ⚠ skipped {src.name}: {r.stderr.decode()[-200:]}')

print(f'Converted: {len(converted)} clips')

# Transcribe with Whisper (base model — fast, good enough for training metadata)
print('\nLoading Whisper base model…')
whisper_model = whisper.load_model('base')

rows = []
for i, wav in enumerate(sorted(converted)):
    print(f'  [{i+1}/{len(converted)}] Transcribing {wav.name}…', end=' ')
    try:
        result = whisper_model.transcribe(str(wav), language='en')
        text = result['text'].strip()
        # Skip clips where Whisper produced no text
        if text:
            rows.append({'audio_file': str(wav), 'text': text, 'speaker_name': 'Pacaveli'})
            print(f'OK ({len(text)} chars)')
        else:
            print('SKIP (empty transcript)')
    except Exception as e:
        print(f'ERROR: {e}')

print(f'\n✅ {len(rows)} clips transcribed')

# Shuffle and split 90/10 train/eval
random.shuffle(rows)
split = max(1, int(len(rows) * 0.9))
train_rows, eval_rows = rows[:split], rows[split:]
if not eval_rows:
    eval_rows = train_rows[:1]  # need at least 1 eval sample

TRAIN_CSV = '/content/train_metadata.csv'
EVAL_CSV  = '/content/eval_metadata.csv'

def write_csv(path, data):
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['audio_file', 'text', 'speaker_name'],
                           delimiter='|')
        w.writeheader()
        w.writerows(data)

write_csv(TRAIN_CSV, train_rows)
write_csv(EVAL_CSV,  eval_rows)
print(f'Train: {len(train_rows)} clips   Eval: {len(eval_rows)} clips')

In [ ]:
# ── Cell 5: Fine-tune XTTS v2 ────────────────────────────────────
# Uses Coqui's built-in XTTS fine-tuning demo trainer.
# The base XTTS v2 model is downloaded automatically (~1.8 GB) on first run.

OUTPUT_PATH = '/content/xtts_finetuned'

try:
    # Coqui TTS >= 0.22 ships an XTTS fine-tuning helper
    from TTS.demos.xtts_ft_demo.utils.gpt_train import train_gpt  # type: ignore

    train_gpt(
        custom_model=None,       # None = start from base XTTS v2
        version='v2',
        language='en',
        num_epochs=6,            # ~30 min on T4 for ~50 clips
        batch_size=4,
        grad_acumm=1,
        train_csv=TRAIN_CSV,
        eval_csv=EVAL_CSV,
        output_path=OUTPUT_PATH,
        max_audio_length=255995, # ~11s at 22kHz
    )
    print('\n✅ Fine-tuning complete!')

except ImportError:
    # Fallback for TTS versions without the demo helper
    print('ℹ️  train_gpt() not found — using Trainer API directly')
    from trainer import Trainer, TrainerArgs  # type: ignore
    from TTS.tts.configs.xtts_config import XttsConfig  # type: ignore
    from TTS.tts.models.xtts import Xtts  # type: ignore
    from TTS.tts.datasets import load_tts_samples  # type: ignore

    config = XttsConfig()
    config.load_json('/root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2/config.json')
    config.output_path = OUTPUT_PATH
    config.epochs = 6
    config.batch_size = 4

    model = Xtts.init_from_config(config)
    model.load_checkpoint(config, checkpoint_dir='/root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2/')

    train_samples, eval_samples = load_tts_samples(
        [{'formatter': 'ljspeech', 'meta_file_train': TRAIN_CSV,
          'meta_file_val': EVAL_CSV, 'path': str(WAV_DIR)}],
        eval_split=True,
    )

    trainer = Trainer(
        TrainerArgs(output_path=OUTPUT_PATH, restore_path=None, skip_train_epoch=False),
        config, output_path=OUTPUT_PATH,
        model=model,
        train_samples=train_samples,
        eval_samples=eval_samples,
    )
    trainer.fit()
    print('\n✅ Fine-tuning complete (Trainer API)')

In [ ]:
# ── Cell 6: Package and download ─────────────────────────────────
import zipfile, shutil
from pathlib import Path
from google.colab import files

out_dir = Path(OUTPUT_PATH)
if not out_dir.exists():
    raise FileNotFoundError(f'Output directory not found: {OUTPUT_PATH}')

# List what was produced
produced = list(out_dir.rglob('*'))
print(f'Output files ({len(produced)}):')
for p in sorted(produced):
    if p.is_file():
        print(f'  {p.relative_to(out_dir)}  ({p.stat().st_size/1e6:.1f} MB)')

# Zip everything
zip_out = '/content/xtts_finetuned.zip'
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in produced:
        if p.is_file():
            zf.write(p, p.relative_to(out_dir))

size_mb = Path(zip_out).stat().st_size / 1e6
print(f'\n📦 Packaged: xtts_finetuned.zip  ({size_mb:.0f} MB)')

# Also save to Drive
drive_dest = '/content/drive/MyDrive/xtts_finetuned.zip'
shutil.copy(zip_out, drive_dest)
print(f'☁️  Saved to Google Drive: {drive_dest}')

# Download to local machine
print('⬇️  Downloading xtts_finetuned.zip …')
files.download(zip_out)

## 📥 Installing the Fine-Tuned Model

After downloading `xtts_finetuned.zip`:

1. Open **AI Vocals Studio** on your local machine
2. Go to the **🎓 Training** tab
3. Click **📥 INSTALL MODEL**
4. Select the downloaded `xtts_finetuned.zip`

The app will extract the checkpoint files to:
```
models/Pacaveli/xtts_finetuned/
```

On the next generation, the engine badge will show: **⚡ XTTS Clone (fine-tuned)**

### What improves after fine-tuning

| Before (zero-shot) | After (fine-tuned) |
|--------------------|--------------------|
| Natural delivery from reference clip | Consistent Pacaveli timbre on every generation |
| Voice varies with reference choice | Stable voice identity regardless of ref |
| Good for most text | Optimized for rap/spoken word delivery |
